# 第4章：Reduce算子与优先队列模拟堆 — 章节介绍

## 1. 实验背景

在深度学习和高性能计算中，**规约操作**（Reduce）是最基础的并行计算原语之一：

- **求和规约**：将 N 个数求和为 1 个数（如 loss 计算）
- **最大值规约**：从 N 个数中找最大值（如 softmax 的分母）
- **TopK 选择**：从 N 个数中选最大的 K 个（如 MoE 路由）

本实验将在华为昇腾平台（310B/910B）上，用 Ascend C 编程框架从零实现这3个算子。

## 2. 前置知识

在开始 ReduceLab 实验之前，建议先了解以下基础知识。

### 2.1 数组与连续存储

Reduce 算子的输入通常以一维或多维数组形式存储。连续内存布局便于批量搬运数据，也有助于提高 NPU 的访存效率。

### 2.2 规约计算

规约计算是将一组输入数据按照指定操作聚合为较少结果的过程，常见操作包括：

- **ReduceSum**：对输入元素求和
- **ReduceMax**：求输入元素最大值
- **TopK**：选择数值最大的 K 个元素

### 2.3 数据分块

由于 Local Memory 容量有限，较大的输入数据通常需要切分为多个 Tile。每个 Tile 分别完成数据搬运、局部计算和结果汇总。

### 2.4 多核并行

输入数据可以分配给多个 AI Core，每个核心处理一部分数据并生成局部结果，最后再完成全局规约。

### 2.5 Ascend C 基础

本实验涉及以下 Ascend C 概念：

- `GlobalTensor`：GM 上的 tensor 视图
- `GetBlockIdx()`：获取当前核编号
- `SetValue` / `GetValue`：GM 上的读写操作
- Tiling 参数：Host 端计算分块策略，通过结构体传递给 Kernel

## 3. 昇腾多核架构（310B/910B）

<div style="text-align: left;">
  <img src="./images/topology.svg" alt="昇腾多核架构示意图" width="720">
</div>

<p style="text-align: left;">昇腾芯片包含 N 个 AI Core，共享 GM (Global Memory)，每个 Core 有独立 UB (Unified Buffer)。</p>

**关键点**：
- 多核并行处理数据（310B=8核，910B=20+核）
- 每个 Core 通过 `GetBlockIdx()` 获取自己的编号
- 数据按 Core 数量切分，每个 Core 处理 1/N 的数据

## 4. 三个算子概览

<table style="text-align: left; margin-left: 0;">
  <thead>
    <tr>
      <th>算子</th>
      <th>功能</th>
      <th>核心算法</th>
      <th>输出</th>
    </tr>
  </thead>
  <tbody>
    <tr>
      <td>ReduceSumLite</td>
      <td>求和规约</td>
      <td>累加器</td>
      <td>1 个标量</td>
    </tr>
    <tr>
      <td>ReduceMaxLite</td>
      <td>最大值规约</td>
      <td>比较</td>
      <td>1 个标量</td>
    </tr>
    <tr>
      <td>TopKReduceLite</td>
      <td>TopK 选择</td>
      <td>小根堆</td>
      <td>K 个值+索引</td>
    </tr>
  </tbody>
</table>

**多核并行策略**：

```
输入 x[1024]:  [0..N-1]  [N..2N-1]  ...  [last]
                ↓         ↓              ↓
Core:          Core0     Core1    ...   CoreN-1
                ↓         ↓              ↓
局部结果:      y[0]      y[1]     ...   y[N-1]
                ↓         ↓              ↓
Host 汇总:     ──────── 最终结果 ────────
```

## 5. 拓扑图

<div style="text-align: left;">
  <img src="./images/topology.svg" alt="Reduce 算子多核并行拓扑图" width="720">
</div>

<p style="text-align: left;">图 1 Reduce 算子多核并行拓扑示意图</p>

## 6. 学习目标

完成本章实验后，你将能够：

1. 理解 Ascend C 编程模型（GM → 计算 → GM 数据流）
2. 掌握多核并行切分策略
3. 理解 Tiling 机制（Host 端计算分块，Kernel 端执行计算）
4. 理解小根堆在 TopK 选择中的应用
5. 完成算子编译、部署和精度验证

## 7. 章节内容

本章由一个实验教程和一个综合实践组成。

### 04.02 Reduce算子与优先队列模拟堆实验

本小节介绍 Reduce 算子的计算特点、工程目录、数据分块方法、Kernel 实现、编译运行过程和结果验证方法。

[进入 04.02 Reduce算子与优先队列模拟堆实验](./04.02_reduce_lab.ipynb)

### 04.03 Reduce算子与优先队列模拟堆章节实践

本小节通过代码补全、ReduceMax 扩展和输入规模调整等任务，检验学习者对规约计算与 Ascend C 算子开发流程的掌握情况。

[进入 04.03 ReduceLab 章节实践](./04.03_chapter_test.ipynb)